In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_Sri_Aurobindo_Marg, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,146.25,243.32,20.71,29.59,32.58,1.45,21.43,NaN,NaN,81.16,1.54,78.02,74.06,989.09,13.55,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,159.70,260.28,16.22,30.42,29.37,1.68,23.10,0.37,2.92,81.16,1.68,82.25,68.46,988.78,13.83,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,193.67,330.29,60.60,41.66,71.43,2.47,31.25,0.41,3.04,79.53,1.02,92.38,67.90,988.69,14.30,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,186.56,336.50,52.56,62.64,76.05,2.99,26.70,0.52,4.85,80.80,1.24,179.97,61.58,987.91,15.12,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,115.80,233.21,15.06,34.57,30.63,1.58,19.73,0.24,1.13,78.41,1.34,203.95,64.96,987.83,14.79,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,298.75,395.75,10.91,39.03,29.63,1.43,42.70,3.44,20.35,78.22,2.32,76.54,99.99,996.83,18.26,0.0,0.0
316,13-11-2025 00:00,14-11-2025 00:00,270.71,376.17,11.15,41.49,31.13,1.38,39.61,3.73,20.93,80.68,2.18,78.33,92.92,997.59,18.07,0.0,0.0
317,14-11-2025 00:00,15-11-2025 00:00,189.06,280.04,9.77,40.68,29.58,1.15,40.41,2.92,11.97,80.02,2.10,83.65,94.58,997.54,18.00,0.0,0.0
318,15-11-2025 00:00,16-11-2025 00:00,213.88,309.08,11.31,40.57,30.59,1.25,42.50,3.23,17.27,79.52,2.10,80.61,96.64,998.05,18.07,0.0,0.0


In [4]:

# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 19)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): []
Dropped rows (>70% NaN): 1
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
SR           0
BP           0
AT           0
RF           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (319, 19)
          From Date           To Date   PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   48.81  243.32  20.71  29.59  32.58   
1  02-01-2025 00:00  03-01-2025 00:00   48.81  260.28  16.22  30.42  29.37   
2  03-01-2025 00:00  04-01-2025 00:00   48.81  330.29  10.24  41.66  23.11   
3  04-01-2025 00:00  05-01-2025 00:00   48.81  336.50  10.24  62.64  23.11   
4  05-01-2025 00:00  06-01-2025 00:00  115.80  233.21  15.06  34.57  30.63   

     CO  Ozone  Benzene  Toluene     RH    WS      WD     SR      BP     AT  \
0  1.45  21.43     0.39    6.195  81.16  1.54   78.02  74.06  989.09  13.55   
1  1.68  23.10     0.37    2.920  81.16  1.68   82.25  68.46  988.78  13.83   
2  0.95  31.25     0.41    3.040  79.53  1.02   92.38  67.90  988.69  14.30   
3  0.95  26.70     0.52    4.850  80.80  1.24  179.97  61.58  987.91  15.12   
4  1.58  19.73     0.24    1.130  78.41  1.34  203.95  64.96  987.83  14.79   

    RF  TOT-RF  
0  0.0     0.0  

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,0.013566,1.556379,1.455134,0.042747,0.893362,1.720134,-0.923543,-0.519794,-0.233441,0.641805,-0.317114,-1.011948,-1.451801,0.779512,-2.274574,0.0,0.0
1,02-01-2025 00:00,03-01-2025 00:00,0.013566,1.795256,0.746934,0.108192,0.549879,2.580083,-0.840906,-0.541460,-0.654763,0.641805,-0.035277,-0.920079,-1.583493,0.757405,-2.225483,0.0,0.0
2,03-01-2025 00:00,04-01-2025 00:00,0.013566,2.781328,-0.196280,0.994462,-0.119965,-0.149322,-0.437615,-0.498127,-0.639325,0.519229,-1.363939,-0.700071,-1.596662,0.750987,-2.143081,0.0,0.0
3,04-01-2025 00:00,05-01-2025 00:00,0.013566,2.868794,-0.196280,2.648727,-0.119965,-0.149322,-0.662765,-0.378961,-0.406472,0.614733,-0.921052,1.202250,-1.745286,0.695362,-1.999316,0.0,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.706246,1.413982,0.563970,0.435418,0.684704,2.206192,-1.007665,-0.682293,-0.885042,0.435005,-0.719739,1.723059,-1.665801,0.689657,-2.057172,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,12-11-2025 00:00,13-11-2025 00:00,0.013566,-0.125903,-0.090602,0.787087,0.577700,1.645355,0.128972,2.784357,1.587567,0.420717,1.253123,-1.044091,-0.842020,1.331477,-1.448799,0.0,0.0
315,13-11-2025 00:00,14-11-2025 00:00,0.013566,-0.125903,-0.052747,0.981057,0.738206,1.458410,-0.023932,-0.519794,1.662183,0.605709,0.971285,-1.005215,-1.008281,1.385675,-1.482111,0.0,0.0
316,14-11-2025 00:00,15-11-2025 00:00,0.013566,2.073570,-0.270412,0.917189,0.572350,0.598460,0.015655,2.221027,0.509499,0.556077,0.810235,-0.889673,-0.969244,1.382109,-1.494384,0.0,0.0
317,15-11-2025 00:00,16-11-2025 00:00,0.013566,2.482591,-0.027511,0.908516,0.680424,0.972351,0.119075,2.556858,1.191332,0.518477,0.810235,-0.955697,-0.920800,1.418479,-1.482111,0.0,0.0


In [10]:
df.to_excel('sriaurobindo2025.xlsx', index=False)